In [6]:
from collections.abc import Callable, Iterable
from typing import Optional
import torch
import math
class AdamW(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, lamd=1e-2, epsilon=1e-8):
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {"lr": lr, "beta1": beta1, "beta2": beta2, "lamd": lamd, "epsilon": epsilon}
        super().__init__(params, defaults)

    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()
        
        for group in self.param_groups: # 这个 param_groups 是参数组，代表模型不同模块的参数可以有不同的学习率
            lr = group["lr"]  # Get the learning rate.
            beta1 = group["beta1"] # 一阶动量平滑系数
            beta2 = group["beta2"] # 二阶动量平滑系数
            lamd = group["lamd"] # weight decay
            epsilon = group["epsilon"] # 分母
            
            
            for p in group["params"]:
                if p.grad is None:
                    continue
                
                # 1. 取state
                state = self.state[p]  # state 是一个map，从参数param映射到字典
                t = state.get("t", 1)  # Get iteration number from the state, or initial value.
                m = state.get("m", torch.zeros_like(p))
                v = state.get("v", torch.zeros_like(p))
                grad = p.grad.data  # Get the gradient of loss with respect to p.
                
                # 2. 更新动量
                m = beta1 * m + (1 - beta1) * m
                v = beta2 * v + (1 - beta1) * v
                
                # 3. 更新参数
                alpha_t = lr * math.sqrt(1 - beta2**t) / (1 - beta1**t)
                p.data -= alpha_t * m / (torch.sqrt(v) + epsilon) * grad
                
                # 4. weight decay(正则化)
                p.data -= lr * lamd * p.data
                
                # 5. 更新状态
                state["t"] = t + 1
                state["m"] = m
                state["v"] = v
        
        return loss


In [7]:
# 初始化参数和优化器
weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
target = torch.diag(torch.ones(10))
opt = AdamW([weights], lr=10)

# 训练循环
for t in range(100):
    opt.zero_grad()  # Reset the gradients for all learnable parameters.
    loss = ((weights - target)**2).mean()  # Compute a scalar loss value.
    print(f"Step {t:3d} | Loss: {loss.cpu().item():.6f}")
    loss.backward()  # Run backward pass, which computes gradients.
    opt.step()  # Run optimizer step.

Step   0 | Loss: 23.326118
Step   1 | Loss: 18.910490
Step   2 | Loss: 15.334095
Step   3 | Loss: 12.437457
Step   4 | Loss: 10.091394
Step   5 | Loss: 8.191280
Step   6 | Loss: 6.652362
Step   7 | Loss: 5.405996
Step   8 | Loss: 4.396581
Step   9 | Loss: 3.579083
Step  10 | Loss: 2.917024
Step  11 | Loss: 2.380859
Step  12 | Loss: 1.946659
Step  13 | Loss: 1.595040
Step  14 | Loss: 1.310305
Step  15 | Loss: 1.079737
Step  16 | Loss: 0.893038
Step  17 | Loss: 0.741866
Step  18 | Loss: 0.619467
Step  19 | Loss: 0.520368
Step  20 | Loss: 0.440138
Step  21 | Loss: 0.375187
Step  22 | Loss: 0.322610
Step  23 | Loss: 0.280051
Step  24 | Loss: 0.245605
Step  25 | Loss: 0.217727
Step  26 | Loss: 0.195168
Step  27 | Loss: 0.176914
Step  28 | Loss: 0.162145
Step  29 | Loss: 0.150198
Step  30 | Loss: 0.140535
Step  31 | Loss: 0.132720
Step  32 | Loss: 0.126401
Step  33 | Loss: 0.121294
Step  34 | Loss: 0.117165
Step  35 | Loss: 0.113830
Step  36 | Loss: 0.111135
Step  37 | Loss: 0.108960
Step  3

Step   0 | Loss: 22.191753
Step   1 | Loss: 14.202724
Step   2 | Loss: 10.469640
Step   3 | Loss: 8.191376
Step   4 | Loss: 6.635015
Step   5 | Loss: 5.501187
Step   6 | Loss: 4.639522
Step   7 | Loss: 3.964604
Step   8 | Loss: 3.423747
Step   9 | Loss: 2.982464
Step  10 | Loss: 2.617139
Step  11 | Loss: 2.311017
Step  12 | Loss: 2.051867
Step  13 | Loss: 1.830546
Step  14 | Loss: 1.640083
Step  15 | Loss: 1.475069
Step  16 | Loss: 1.331250
Step  17 | Loss: 1.205232
Step  18 | Loss: 1.094280
Step  19 | Loss: 0.996166
Step  20 | Loss: 0.909058
Step  21 | Loss: 0.831441
Step  22 | Loss: 0.762047
Step  23 | Loss: 0.699813
Step  24 | Loss: 0.643840
Step  25 | Loss: 0.593363
Step  26 | Loss: 0.547729
Step  27 | Loss: 0.506376
Step  28 | Loss: 0.468821
Step  29 | Loss: 0.434644
Step  30 | Loss: 0.403482
Step  31 | Loss: 0.375016
Step  32 | Loss: 0.348967
Step  33 | Loss: 0.325091
Step  34 | Loss: 0.303172
Step  35 | Loss: 0.283021
Step  36 | Loss: 0.264467
Step  37 | Loss: 0.247362
Step  38 

In [3]:
weights

Parameter containing:
tensor([[ 1.0883e+00, -9.8148e-02, -7.5250e-02, -1.5428e-02, -1.2149e-03,
          9.0027e-02,  1.0008e-01,  1.6763e-01, -6.8805e-02, -1.7817e-01],
        [-1.8750e-01,  8.9158e-01,  2.0193e-02,  1.3365e-01, -1.7304e-01,
          6.6387e-02, -5.6824e-02, -1.0839e-01, -1.4112e-01, -5.8116e-02],
        [-4.5103e-02,  6.2086e-02,  9.7641e-01, -5.1490e-02,  1.1562e-01,
         -7.3214e-02, -8.5030e-02, -5.0957e-02,  2.3873e-03, -5.0906e-02],
        [-5.0225e-03,  1.1320e-01, -9.6118e-02,  1.0696e+00, -5.8038e-03,
          1.3680e-01,  1.5120e-02, -1.6072e-01,  4.9620e-03,  9.7884e-02],
        [-3.5379e-02,  1.9232e-01,  2.0233e-01,  9.0134e-02,  9.7737e-01,
          1.1923e-01,  2.3431e-02,  6.6110e-02, -2.0026e-02, -9.7393e-02],
        [ 1.6971e-01,  1.9947e-01, -2.4471e-02,  1.2425e-01, -6.6749e-02,
          1.0084e+00,  9.5492e-02,  2.3273e-01,  9.0563e-02, -2.2331e-02],
        [-7.5774e-02,  2.9650e-02, -1.3659e-01, -8.0875e-02, -8.4958e-02,
         -